In [3]:
"""
Real-World Pipeline: AI Research Team
Uses Groq (OpenAI-compatible client) + Shared Memory concept
Agents: Search → Analyst → Writer → QA
"""

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

# ─── Groq Client (OpenAI-compatible) ──────────────────────────────────────────
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# ─── LLM Helper ───────────────────────────────────────────────────────────────
def call_llm(system_prompt: str, user_message: str) -> str:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content.strip()


# ─── Shared Memory (Message Bus) ──────────────────────────────────────────────
class SharedMemory:
    def __init__(self):
        self._store = {}

    def write(self, key: str, value: str):
        self._store[key] = value
        print(f"  [Memory] '{key}' updated ({len(value)} chars)")

    def read(self, key: str) -> str:
        return self._store.get(key, "")


# ─── Agents ───────────────────────────────────────────────────────────────────
def search_agent(goal: str, memory: SharedMemory):
    print("\n[SearchAgent] Finding data...")
    result = call_llm(
        "You find and list key facts and trends about the given topic. Be concise.",
        f"Find key data and trends for: {goal}"
    )
    memory.write("search", result)

def analyst_agent(memory: SharedMemory):
    print("\n[AnalystAgent] Analysing...")
    result = call_llm(
        "You interpret raw facts and summarise them into key insights. Be concise.",
        f"Analyse this data:\n{memory.read('search')}"
    )
    memory.write("analysis", result)

def writer_agent(memory: SharedMemory):
    print("\n[WriterAgent] Drafting report...")
    result = call_llm(
        "You draft a short professional report from the insights provided.",
        f"Write a report based on:\n{memory.read('analysis')}"
    )
    memory.write("draft", result)

def qa_agent(memory: SharedMemory):
    print("\n[QAAgent] Reviewing...")

    threshold = float(input("Enter quality threshold (0.0 - 1.0): "))
    max_retries = int(input("Enter max retries: "))

    for attempt in range(1, max_retries + 1):
        print(f"\n  [QAAgent] Attempt {attempt}/{max_retries}")

        # Step 1: Score the draft
        score_response = call_llm(
            "You are a strict report evaluator. Given a report, return ONLY a float score between 0.0 and 1.0 based on tone, facts, and grammar. Nothing else.",
            f"Score this report:\n{memory.read('draft')}"
        )

        score = float(score_response.strip())
        print(f"  [QAAgent] Quality Score: {score}")

        # Step 2: Check threshold
        if score > threshold:
            print(f"  [QAAgent] Score {score} > {threshold} Finalizing report...")
            result = call_llm(
                "You proofread the report: fix tone, facts, and grammar. Return the final polished version.",
                f"Review and polish this report:\n{memory.read('draft')}"
            )
            memory.write("final_report", result)
            return

        # Step 3: Improve the draft and retry
        print(f"  [QAAgent] Score {score} <= {threshold}  Improving draft...")
        improved = call_llm(
            "You improve the report's tone, facts, and grammar. Return only the improved report.",
            f"Improve this report:\n{memory.read('draft')}"
        )
        memory.write("draft", improved)

    print(f"\n  [QAAgent] Max retries reached. Saving best version.")
    memory.write("final_report", memory.read("draft"))


# ─── Pipeline ─────────────────────────────────────────────────────────────────
def run_pipeline(goal: str):
    memory = SharedMemory()

    search_agent(goal, memory)
    analyst_agent(memory)
    writer_agent(memory)
    qa_agent(memory)

    print("FINAL REPORT")
    print(memory.read("final_report"))

    return memory


# ─── Main ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    GOAL = "Write a comprehensive market report on EV industry trends in India for 2025."
    run_pipeline(GOAL)


[SearchAgent] Finding data...
  [Memory] 'search' updated (2310 chars)

[AnalystAgent] Analysing...
  [Memory] 'analysis' updated (827 chars)

[WriterAgent] Drafting report...
  [Memory] 'draft' updated (2574 chars)

[QAAgent] Reviewing...

  [QAAgent] Attempt 1/3
  [QAAgent] Quality Score: 0.92
  [QAAgent] Score 0.92 <= 0.98  Improving draft...
  [Memory] 'draft' updated (3960 chars)

  [QAAgent] Attempt 2/3
  [QAAgent] Quality Score: 0.92
  [QAAgent] Score 0.92 <= 0.98  Improving draft...
  [Memory] 'draft' updated (4320 chars)

  [QAAgent] Attempt 3/3
  [QAAgent] Quality Score: 0.92
  [QAAgent] Score 0.92 <= 0.98  Improving draft...
  [Memory] 'draft' updated (5994 chars)

  [QAAgent] Max retries reached. Saving best version.
  [Memory] 'final_report' updated (5994 chars)
FINAL REPORT
**Electric Vehicle Market Report: India**

**Executive Summary:**

The Indian electric vehicle (EV) market is poised for exponential growth, with a projected valuation of $7.9 billion by 2025, growing